# 01 — Model, units and numerical operators

This notebook connects the model to the actual shared engine. It runs an analytical free-packet comparison and plots both kinetic dispersions. Review the mathematical model and methods in **README.md**.

The engine solves \(i\partial_t\psi=(-\nabla^2/2+V)\psi\), using the grid-weighted norm. The Gaussian \(\sigma\) is the density SD. All inputs below are dimensionless; electron units are a display conversion.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "wavepacket").is_dir())
print("Python:", sys.executable)

OUT = ROOT / "results" / "notebook_runs" / "01_model_and_methods"
OUT.mkdir(parents=True, exist_ok=True)
from wavepacket.config import SimulationConfig
from wavepacket.model import make_grid, initial_state
from wavepacket.operators import potential, kinetic_symbol
from wavepacket.observables import measure, state_errors
from wavepacket.units import electron_scales
from wavepacket.validation import propagate, analytic_free

config = SimulationConfig()
grid = make_grid(config.grid)
initial = initial_state(config, grid)
final, _, diagnostics = propagate(config)
truth = analytic_free(grid, [-10.0], [2.0], [2.0], config.solver.duration)
errors = state_errors(final, truth, grid)
assert errors["raw_l2"] < 1e-10
print({"diagnostics": diagnostics, "errors": errors, "electron_scales_L0_1nm": electron_scales(1).as_dict()})
(OUT / "model_evidence.json").write_text(json.dumps({"config": config.model_dump(mode="json"), "diagnostics": diagnostics, "errors": errors}, indent=2))


In [ ]:
x = grid.axes[0]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(x, abs(initial)**2, label="initial")
axes[0].plot(x, abs(final)**2, label="numerical final")
axes[0].plot(x, abs(truth)**2, "--", label="analytic final")
axes[0].set(xlabel="x / L0", ylabel="probability density", xlim=(-20, 10))
axes[0].legend()
k = 2 * np.pi * np.fft.fftfreq(grid.points[0], grid.spacing[0])
order = np.argsort(k)
axes[1].plot(k[order], kinetic_symbol(grid, "spectral")[order], label="spectral")
axes[1].plot(k[order], kinetic_symbol(grid, "fd2")[order], label="FD2")
axes[1].set(xlabel="k L0", ylabel="kinetic energy / E0")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT / "model_and_dispersion.png", dpi=150)
plt.close(fig)
display(Image(filename=str(OUT / "model_and_dispersion.png")))


## Review questions

1. Derive \(\lambda_{\rm FD2}=2\sin^2(k\Delta x/2)/\Delta x^2\). Why do these dispersion curves differ even at equal timestep?
2. Derive \(\sigma_x(t)^2=\sigma^2+t^2/(4\sigma^2)\). Check the measured variance below.
3. Why must \(E_0=\hbar^2/(m_e L_0^2)\) rather than half that value with this engine's equation?
4. Why does comparing free evolution alone fail to test splitting commutators?

The exact reference is on the line. The numerical result approximates it here because packet tails and wraparound are negligible.

In [ ]:
metrics = measure(final, grid, potential(config, grid))
expected_variance = 4 + config.solver.duration**2 / 16
assert np.isclose(metrics["x_variance"], expected_variance, atol=1e-10)
print({"measured_variance": metrics["x_variance"], "analytic_variance": expected_variance,
       "uncertainty_product": metrics["x_px_uncertainty"], "xp_covariance": metrics["x_px_covariance"]})
